## 加载文档

In [2]:
import os
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
# from langchain.document_loaders import UnstructuredPDFLoader
from langchain_community.document_loaders import UnstructuredPDFLoader

file_path = "./机器学习常用数据集.pdf"
print('start load data')
loader = UnstructuredPDFLoader(file_path)
docs = loader.load()
print('end load data')
print(len(docs))

start load data
end load data
1


In [3]:
langchain_community.__version__

'0.0.38'

In [4]:
docs

[Document(metadata={'source': './机器学习常用数据集.pdf'}, page_content='AI 从业者都应该知道的实验数据集\n\n本文作者：黄善清\n\n2018-11-03 17:27\n\n数据集对于深度学习模型的重要性不言而喻，然而根据性质、类型、领域的不同，数据集往往散落在不同的资源平\n\n台里，急需人们做出整理。 fast.ai 近期将这些重要的数据集汇总到了一篇文章里， AI 科技评论把文章编译如下。\n\n少了数据，我们的机器学习和深度学习模型什么也干不了。这么说吧，那些创建了数据集、让我们可以训练模型的\n\n人，都是我们的英雄，虽然这些人常常并没有得到足够的感谢。让人庆幸的是，那批最有价值的数据集后来成了\n\n「学术基准线」——被研究人员广泛引用，尤其在算法变化的对比上；不少名字则成为圈内外都耳熟能详的名\n\n称，如 MNIST、CIFAR 10 以及 Imagenet 等。\n\n身为 fast.ai 的一员，我们自觉欠这些数据集的创建者一句真挚的感谢，所以我们决定，通过与 AWS 合作，把一\n\n些最重要的数据集集中整理在一处，数据集自身采用标准格式，存储服务器也是快速的、可靠的（请参阅下方的完\n\n整列表与链接）。如果您在研究中使用了这些数据集，我们希望您记得引用原始论文（我们已经在表单中提供引用\n\n链接）；如果您将它们用作商业或教育项目的一部分，请考虑添加致谢文及数据集原链接。\n\n我们之所以经常在教学中引用这些数据集，是因为它们就是学生们很有可能遇到的数据类型的绝佳例子，此外，学\n\n生可以将自己的工作与引用这些数据集的学术成果进行对比，从而取得进步。此外，我们也会使用 Kaggle\n\nCompetitions 数据集，Kaggle 的 public leaderboards 允许学生在世界最好的数据集里测试自己的模型，不过\n\nKaggle 数据集并不会在本次表单中出现。\n\n图像分类领域\n\n1）MNIST\n\n经典的小型（28x28 像素）灰度手写数字数据集，开发于 20 世纪 90 年代，主要用于测试当时最复杂的模型；到\n\n了今日，MNIST 数据集更多被视作深度学习的基础教材。fast.ai 版本的数据集舍弃了原始的特殊二进制格式，转\n\n而采用标准的 

In [5]:
# 加载多个文件
# pdf 文件/文件的位置。
pdf_folder_path = './data_test'
loaders = [UnstructuredPDFLoader(os.path.join(pdf_folder_path, fn)) for fn in os.listdir(pdf_folder_path) if fn.endswith('.pdf')]


In [6]:
len(loaders)

2

In [7]:
print(docs[0].page_content[0:100])
print(docs[0].metadata)

AI 从业者都应该知道的实验数据集

本文作者：黄善清

2018-11-03 17:27

数据集对于深度学习模型的重要性不言而喻，然而根据性质、类型、领域的不同，数据集往往散落在不同的资源平

台
{'source': './机器学习常用数据集.pdf'}


# 使用 RAG 进行问答

In [8]:
# import getpass
# import os

# os.environ["OPENAI_API_KEY"] = getpass.getpass()

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4o")


import os
from langchain_ollama import ChatOllama

from langchain_ollama import OllamaEmbeddings


llm = ChatOllama(
    model="qwen2.5:14b",
    base_url="http://localhost:11434",
    # model="qwen2.5:14b",
    # temperature=0,
    # other params...
)

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = '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'


## 文档切割

In [9]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# splits = text_splitter.split_documents(docs)

# 加载多个文档
all_documents = []
for loader in loaders:
    print("Loading raw document..., it may take long time " + loader.file_path)
    raw_documents = loader.load()
    print('load ok')
    print("Splitting text...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        length_function=len,
    )
    documents = text_splitter.split_documents(raw_documents)
    all_documents.extend(documents)

Loading raw document..., it may take long time ./data_test/2023-医械法规文件汇编（下册）.pdf
Splitting text...
Loading raw document..., it may take long time ./data_test/2023-医械法规文件汇编（上册）.pdf
Splitting text...


In [10]:
all_documents[:2]

[Document(metadata={'source': './data_test/2023-医械法规文件汇编（下册）.pdf'}, page_content='医疗器械法规文件汇编\n\n下 册\n\n广东省药品监督管理局 印制\n\n2023 年 11 月\n\n目 录（下册）\n\n一、工作文件\n\n1．关于发布医疗器械唯一标识系统规则的公告\n\n（2019 年 第 66 号）........................................................................................................................2\n\n2．关于做好第三批实施医疗器械唯一标识工作的公告\n\n（2023 年 第 22 号）........................................................................................................................5\n\n3．关于发布医疗器械安全和性能基本原则的通告'),
 Document(metadata={'source': './data_test/2023-医械法规文件汇编（下册）.pdf'}, page_content='3．关于发布医疗器械安全和性能基本原则的通告\n\n（2020 年 第 18 号）......................................................................................................................12\n\n4．关于发布医疗器械通用名称命名指导原则的通告\n\n(2019 年 第 99 号).......................................................................................................................... 21\n\n5．关于发布医疗器械产品适用强制性标准清单的通告\n\n（2022 年 第 42 号）........................

In [12]:
len(raw_documents[0].page_content)

499903

## 向量化

In [16]:
vectorstore = Chroma.from_documents(documents=all_documents, embedding=OllamaEmbeddings(model='qwen2.5:14b', base_url='http://125.69.16.175:11434'))
# vectorstore = Chroma.from_documents(documents=all_documents, embedding=OllamaEmbeddings(model='qwen2.5:14b'))

retriever = vectorstore.as_retriever()

------------------------
http://125.69.16.175:11434/api/embed
------------------------


## 向量持久化

In [21]:
vectorstore.persist?

Object `vectorstore.persist` not found.


## RAG问答

In [18]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "你是一个负责答疑任务的助手。"
    "使用以下检索到的上下文来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "最多使用三个句子并保持答案简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

results = rag_chain.invoke({"input": "医疗器械监督管理条例"})
results



------------------------
http://125.69.16.175:11434/api/embed
------------------------
------------------------
http://125.69.16.175:11434/api/chat
------------------------


{'input': '医疗器械监督管理条例',
 'context': [Document(metadata={'source': './data_test/2023-医械法规文件汇编（上册）.pdf'}, page_content='法通过国家企业信用信息公示系统向社会公示。\n\n第三十二条 广告审查机关的工作人员玩忽职守、滥用职权、徇私舞弊的，依法给予处\n\n分。构成犯罪的，依法追究刑事责任。\n\n第三十三条 本办法涉及的文书格式范本由国家市场监督管理总局统一制定。 第三十四条 本办法自 2020 年 3 月 1 日起施行。1996 年 12 月 30 日原国家工商行政管 理局令第 72 号公布的《食品广告发布暂行规定》，2007 年 3 月 3 日原国家工商行政管理总 局、原国家食品药品监督管理局令第 27 号公布的《药品广告审查发布标准》，2007 年 3 月 13 日原国家食品药品监督管理局、原国家工商行政管理总局令第 27 号发布的《药品广告审 查办法》，2009 年 4 月 7 日原卫生部、原国家工商行政管理总局、原国家食品药品监督管理\n\n局令第 65 号发布的《医疗器械广告审查办法》，2009 年 4 月 28 日原国家工商行政管理总局、 原卫生部、原国家食品药品监督管理局令第 40 号公布的《医疗器械广告审查发布标准》同时 废止。\n\n104\n\n《医疗器械生产监督管理办法》'),
  Document(metadata={'source': './data_test/2023-医械法规文件汇编（上册）.pdf'}, page_content='（三）行政相对人收到检验报告 12 个工作日后，国家抽检系统仍未收到《异议申诉收到\n\n回执》等相关材料的，该批产品异议申诉材料上传通道予以关闭。\n\n（四）相关省级药品监督管理部门应当在收到该异议申诉书面申请后 15 个工作日内进行 调查核实、确认核实结果、提出处理建议，出具公文并将扫描件上传至国家抽检系统。相关 省级药品监督管理部门调查核实后，对相关异议申诉不予支持的，应当向申请人出具公文， 并将扫描件上传至国家抽检系统。\n\n相关省级药品监督管理部门未进行调查核实、未确认核实结果和提出处理建议的，有关\n\n材料不予办理。\n\n（五）中检院技术监督中心应当组织开展异议申诉专家审议

In [22]:
q2 = '医疗器械唯一标识系统规则有几条规则？'
results = rag_chain.invoke({"input": q2})
results

------------------------
http://127.0.0.1:11434/api/embed
------------------------
------------------------
http://127.0.0.1:11434/api/chat
------------------------


{'input': '医疗器械唯一标识系统规则有几条规则？',
 'context': [Document(metadata={'source': './data_test/2023-医械法规文件汇编（下册）.pdf'}, page_content='01 认知言语视听障碍康复设备\n\n十五、22 临床检验器械\n\n一级产品类别\n\n01 血液学分析设备\n\n11 采样设备和器具\n\n二级产品类别\n\n03 视觉治疗设备\n\n01 眼科激光诊断设备\n\n04 眼科冷冻治疗设备\n\n06 眼科治疗和手术辅助器具\n\n14 义眼片\n\n15 人工晶状体、人工玻璃体 植入器械\n\n16 囊袋张力环植入器械\n\n二级产品类别\n\n05 妇产科用扩张器、牵开器\n\n10 子宫输卵管造影、输卵管 通液器械\n\n02 妇科假体器械\n\n01 辅助生殖导管\n\n02 辅助生殖穿刺取卵/精针\n\n03 辅助生殖微型工具\n\n二级产品类别\n\n07 助听器\n\n二级产品类别\n\n02 血细胞分析仪器\n\n04 静脉血样采血管\n\n11\n\n管理类别\n\nII\n\nII 类部分\n\nII\n\nII 类部分\n\nII 类部分\n\nII 类部分\n\nII 类部分\n\n管理类别\n\nII 类部分\n\nII 类部分\n\nII 类部分\n\nII\n\nII\n\nII\n\n管理类别\n\nII\n\n管理类别\n\nII\n\nII\n\n关于发布医疗器械安全和性能基本原则的通告\n\n（2020 年 第 18 号）'),
  Document(metadata={'source': './data_test/2023-医械法规文件汇编（下册）.pdf'}, page_content='4. 审查发布 标准送审稿经医疗器械标准化技术委员会、医疗器械标准管理中心、国家局及国务院标\n\n准化行政主管部门（仅国标）审查，通过后发布，并按相关规定进行公开，供公众查阅。\n\n5. 复审废止 医疗器械标准化技术委员会对已发布实施的医疗器械标准开展复审工作，复审结论分为\n\n继续有效、修订或者废止。对复审结论为废止的医疗器械行业标准以公告形式发布。\n\n三、医疗器械标准的查询方法\n\n截至 2022 年 2 月 18 日，由